In [3]:
import re
import pandas as pd
import matplotlib.pyplot as plt

# ========= Paden naar je bestanden (jouw input) =========
path_emissies = "Emissies_Nederland__wegverkeer_13102025_110220.csv"
path_km       = "Verkeer_motorvoertuigen_13102025_105929.csv"

# ========= Helpers =========
def find_col(df, startswith):
    """Zoek kolom die met 'startswith' begint (CBS wisselt soms subtitels)."""
    for c in df.columns:
        if str(c).startswith(startswith):
            return c
    return None

def plot_line(years, values, title, ylabel):
    """Eenvoudige lijngrafiek met 1 y-as."""
    plt.figure()
    plt.plot(years, values, marker='o', linewidth=2)
    plt.title(title)
    plt.xlabel("Jaar")
    plt.ylabel(ylabel)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()

# ========= 1) Inlezen CSV’s =========
df_e = pd.read_csv(path_emissies)
df_km = pd.read_csv(path_km)

# ========= 2) Emissies voorbereiden (personenauto’s) =========
# Vind kolommen voor CO2/NOx/PM10 (mln kg)
co2_col  = find_col(df_e, "Emissies/Kooldioxide (CO2)")
nox_col  = find_col(df_e, "Emissies/Stikstofoxiden (NOx)")
pm10_col = find_col(df_e, "Emissies/PM10 Totaal")
if not all([co2_col, nox_col, pm10_col]):
    raise ValueError("Kon CO₂/NOx/PM10-kolommen niet vinden in het emissies-bestand.")

# Filter personenauto's (let op apostrof) en pak jaren 2018–2023
d_e = df_e[df_e['Voertuigtype'] == "Personenauto's"].copy()
d_e['Jaar'] = d_e['Perioden'].astype(str).str.extract(r'(\d{4})').astype(int)
d_e = d_e[(d_e['Jaar'] >= 2018) & (d_e['Jaar'] <= 2023)]

# Houd relevante kolommen en hernoem
d_e = d_e[['Jaar', co2_col, nox_col, pm10_col]].rename(columns={
    co2_col:  'CO2_mlnkg',
    nox_col:  'NOx_mlnkg',
    pm10_col: 'PM10_mlnkg'
}).sort_values('Jaar')

# ========= 3) KILOMETERS uit 2e CSV (Personenauto, totaal in NL) =========
# In tweede CSV heet het doorgaans 'Voertuigtypes' en 'Personenauto' (zonder 's)
km_total_col = find_col(df_km, "Kilometers in Nederland/Totaal kilometers in Nederland")
if not km_total_col:
    raise ValueError("Kon kolom 'Kilometers in Nederland/Totaal kilometers in Nederland (x mln km)' niet vinden in km-bestand.")

d_km = df_km[df_km['Voertuigtypes'] == 'Personenauto'].copy()
d_km['Jaar'] = d_km['Perioden'].astype(str).str.extract(r'(\d{4})').astype(int)
d_km = d_km[['Jaar', km_total_col]].rename(columns={km_total_col: 'km_mln'}).sort_values('Jaar')

# ========= 4) EERST: DRIE LOSSE PLOTS – ALLEEN TOTAAL (mln kg) =========
plot_line(d_e['Jaar'], d_e['CO2_mlnkg'], "CO₂-emissies personenauto’s (totaal)", "CO₂ (mln kg)")
plot_line(d_e['Jaar'], d_e['NOx_mlnkg'], "NOx-emissies personenauto’s (totaal)", "NOx (mln kg)")
plot_line(d_e['Jaar'], d_e['PM10_mlnkg'], "PM10-emissies personenauto’s (totaal)", "PM10 (mln kg)")

# ========= 5) Samenvoegen met kilometers + intensiteiten (g/km) =========
df = d_e.merge(d_km, on='Jaar', how='inner')

# g/km = (mln kg * 1000) / (mln km)
df['CO2_g_per_km']  = (df['CO2_mlnkg']  * 1000) / df['km_mln']
df['NOx_g_per_km']  = (df['NOx_mlnkg']  * 1000) / df['km_mln']
df['PM10_g_per_km'] = (df['PM10_mlnkg'] * 1000) / df['km_mln']

# ========= 6) Toon tabel (kilometers, totalen, intensiteiten) =========
print("\nTabel: kilometers (mln km), totalen (mln kg) en intensiteiten (g/km)")
print(df[['Jaar', 'km_mln', 'CO2_mlnkg', 'NOx_mlnkg', 'PM10_mlnkg',
          'CO2_g_per_km', 'NOx_g_per_km', 'PM10_g_per_km']].to_string(index=False))

# ========= 7) Tot slot: DRIE LOSSE PLOTS – ALLEEN INTENSITEIT (g/km) =========
plot_line(df['Jaar'], df['CO2_g_per_km'],  "CO₂-intensiteit personenauto’s (g/km)",  "CO₂ (g/km)")
plot_line(df['Jaar'], df['NOx_g_per_km'],  "NOx-intensiteit personenauto’s (g/km)",  "NOx (g/km)")
plot_line(df['Jaar'], df['PM10_g_per_km'], "PM10-intensiteit personenauto’s (g/km)", "PM10 (g/km)")


FileNotFoundError: [Errno 2] No such file or directory: 'Emissies_Nederland__wegverkeer_13102025_110220.csv'

In [ ]:
print('hello Twan')

hello world
